# ORC Basics — 07: Complex Types in ORC

ORC natively supports struct, list, map, and union types.
These are stored as nested column streams within the stripe.

Topics: StructType, ArrayType, MapType in ORC, nested column pruning, explode.


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/orc_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('orc-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 13:28:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2


## Step 1 — StructType in ORC



In [2]:

nested_schema = StructType([
    StructField("order_id", LongType()),
    StructField("customer", StructType([
        StructField("id",      LongType()),
        StructField("name",    StringType()),
        StructField("country", StringType()),
        StructField("tier",    StringType()),
    ])),
    StructField("amount",    DoubleType()),
    StructField("status",    StringType()),
])
nested_data = spark.createDataFrame([
    (i, (i, f"Customer_{i}", ["US","UK","DE","FR"][i%4], ["gold","silver","bronze"][i%3]),
     float(i*50), "confirmed")
    for i in range(100_000)
], nested_schema)
NESTED_PATH = f"{DATA_DIR}/nested_orc"
nested_data.write.mode("overwrite").orc(NESTED_PATH)
print("Nested ORC written:")
df_n = spark.read.orc(NESTED_PATH)
df_n.printSchema()
df_n.show(5, truncate=False)


26/04/10 13:28:28 WARN TaskSetManager: Stage 0 contains a task of very large size (1060 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Nested ORC written:
root
 |-- order_id: long (nullable = true)
 |-- customer: struct (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- tier: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)

+--------+---------------------------+------+---------+
|order_id|customer                   |amount|status   |
+--------+---------------------------+------+---------+
|0       |{0, Customer_0, US, gold}  |0.0   |confirmed|
|1       |{1, Customer_1, UK, silver}|50.0  |confirmed|
|2       |{2, Customer_2, DE, bronze}|100.0 |confirmed|
|3       |{3, Customer_3, FR, gold}  |150.0 |confirmed|
|4       |{4, Customer_4, US, silver}|200.0 |confirmed|
+--------+---------------------------+------+---------+
only showing top 5 rows


## Step 2 — Nested Column Pruning in ORC



In [3]:

# ORC supports struct field pruning just like Parquet
t_all, t_pruned = [], []
for _ in range(3):
    t0=time.time(); spark.read.orc(NESTED_PATH).count(); t_all.append(time.time()-t0)
    t0=time.time(); spark.read.orc(NESTED_PATH).select("order_id","customer.country","amount").count(); t_pruned.append(time.time()-t0)
print(f"Read all struct fields: {sorted(t_all)[1]:.3f}s")
print(f"Read customer.country only: {sorted(t_pruned)[1]:.3f}s  (struct field pruning)")


Read all struct fields: 0.360s
Read customer.country only: 0.346s  (struct field pruning)


## Step 3 — ArrayType in ORC



In [4]:

arr_schema = StructType([
    StructField("product_id", LongType()),
    StructField("tags",    ArrayType(StringType())),
    StructField("ratings", ArrayType(DoubleType())),
])
arr_data = spark.createDataFrame([
    (1, ["tech","premium","laptop"], [4.5,5.0,4.0]),
    (2, ["budget","phone"],          [3.5,4.0]),
    (3, ["furniture","ergonomic"],   [5.0,4.5,3.5,4.0]),
], arr_schema)
ARR_PATH = f"{DATA_DIR}/array_orc"
arr_data.write.mode("overwrite").orc(ARR_PATH)
df_arr = spark.read.orc(ARR_PATH)
df_arr.show(truncate=False)

# Explode array
print("Explode tags:")
df_arr.select("product_id", F.explode("tags").alias("tag")).show()

# Array functions
print("Array aggregations:")
df_arr.select("product_id",
    F.size("tags").alias("tag_count"),
    F.array_contains("tags","tech").alias("is_tech"),
    F.aggregate("ratings", F.lit(0.0), lambda acc,x: acc+x).alias("rating_sum"),
).show()


+----------+-----------------------+--------------------+
|product_id|tags                   |ratings             |
+----------+-----------------------+--------------------+
|3         |[furniture, ergonomic] |[5.0, 4.5, 3.5, 4.0]|
|1         |[tech, premium, laptop]|[4.5, 5.0, 4.0]     |
|2         |[budget, phone]        |[3.5, 4.0]          |
+----------+-----------------------+--------------------+

Explode tags:
+----------+---------+
|product_id|      tag|
+----------+---------+
|         3|furniture|
|         3|ergonomic|
|         1|     tech|
|         1|  premium|
|         1|   laptop|
|         2|   budget|
|         2|    phone|
+----------+---------+

Array aggregations:
+----------+---------+-------+----------+
|product_id|tag_count|is_tech|rating_sum|
+----------+---------+-------+----------+
|         3|        2|  false|      17.0|
|         1|        3|   true|      13.5|
|         2|        2|  false|       7.5|
+----------+---------+-------+----------+



## Step 4 — MapType in ORC



In [5]:

map_schema = StructType([
    StructField("event_id",   LongType()),
    StructField("properties", MapType(StringType(), StringType())),
    StructField("metrics",    MapType(StringType(), DoubleType())),
])
map_data = spark.createDataFrame([
    (1, {"page":"/home","device":"mobile"},  {"duration":45.2,"scroll":78.0}),
    (2, {"page":"/cart","device":"desktop"}, {"duration":120.5,"scroll":100.0}),
    (3, {"page":"/search"},                  {"duration":15.0}),
], map_schema)
MAP_PATH = f"{DATA_DIR}/map_orc"
map_data.write.mode("overwrite").orc(MAP_PATH)
df_map = spark.read.orc(MAP_PATH)
df_map.show(truncate=False)

print("Map access:")
df_map.select("event_id",
    F.col("properties").getItem("page").alias("page"),
    F.col("metrics").getItem("duration").alias("duration"),
    F.map_keys("properties").alias("prop_keys"),
).show(truncate=False)


+--------+----------------------------------+------------------------------------+
|event_id|properties                        |metrics                             |
+--------+----------------------------------+------------------------------------+
|1       |{device -> mobile, page -> /home} |{duration -> 45.2, scroll -> 78.0}  |
|2       |{device -> desktop, page -> /cart}|{duration -> 120.5, scroll -> 100.0}|
|3       |{page -> /search}                 |{duration -> 15.0}                  |
+--------+----------------------------------+------------------------------------+

Map access:
+--------+-------+--------+--------------+
|event_id|page   |duration|prop_keys     |
+--------+-------+--------+--------------+
|1       |/home  |45.2    |[device, page]|
|2       |/cart  |120.5   |[device, page]|
|3       |/search|15.0    |[page]        |
+--------+-------+--------+--------------+



## Summary



In [6]:

print("""
Complex types in ORC — same Spark API as Parquet:
  Struct: df.select("parent.child")
  Array:  F.explode("col"), F.size("col"), F.array_contains("col", val)
  Map:    F.col("map").getItem("key"), F.explode("map"), F.map_keys("map")

Nested column pruning:
  ORC supports struct field pruning (reads only needed nested fields)
  df.select("customer.country")  ← reads only country bytes in ORC file

Storage: each nested type has its own column stream(s) in the ORC stripe.
  Struct: one stream per field
  List:   length stream + element stream
  Map:    length stream + key stream + value stream
""")



Complex types in ORC — same Spark API as Parquet:
  Struct: df.select("parent.child")
  Array:  F.explode("col"), F.size("col"), F.array_contains("col", val)
  Map:    F.col("map").getItem("key"), F.explode("map"), F.map_keys("map")

Nested column pruning:
  ORC supports struct field pruning (reads only needed nested fields)
  df.select("customer.country")  ← reads only country bytes in ORC file

Storage: each nested type has its own column stream(s) in the ORC stripe.
  Struct: one stream per field
  List:   length stream + element stream
  Map:    length stream + key stream + value stream

